In [7]:
def load_config(path: str = "train_config.json") -> dict:
    with open(path) as f:
        return json.load(f)

In [6]:
## TO-DO: 
# 1. implementar variação quantized

## Authors approach 

To run this notebook, you'll have to clone authors' repo in the root.

In this section, ill follow as closely as possible, but adapting the code to the notebook format. 
The original author script was intended to run via bash script.

The authors used a PEFT fork local version, and this is what i'll do for this section.

⚠️ The code is not adapted to run in parallel with multiple GPUs. 

⚠️ The code is not adapted to run Llama < 3


In [1]:
import os

In [2]:
import sys

In [3]:
import torch
import transformers
from datasets import load_dataset

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, AutoModel  # noqa: F402

In [4]:
sys.path.insert(0, "/teamspace/studios/this_studio/DoRA/commonsense_reasoning/peft/src")

In [6]:
from peft import (  # noqa: E402 /// This should be replaced for last version, since it's already available in PEFT.
    LoraConfig,
    DoraConfig,
    BottleneckConfig, #unused
    PrefixTuningConfig, #unused
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_int8_training,
    set_peft_model_state_dict,
)

### Parameter setting

In [7]:
# base / data / output
base_model = "meta-llama/Meta-Llama-3-8B" # script
data_path = "commonsense_170k.json" # script
output_dir = "./finetuned_result/r32_lr1e-4" # script ($3)
adapter_name = "dora" # script
load_8bit = False # default

# training hyperparams
batch_size = 16 # script, in paper
micro_batch_size = 16 # script
num_epochs = 3 # script, in paper
learning_rate = 1e-4 # script, in paper
weight_decay = 0.0 # default
cutoff_len = 256 # script
val_set_size = 120 # script
use_gradient_checkpointing = True # script (flag)
eval_step = 80 # script
save_step = 80 # script

# lora hyperparams
lora_r = 32 # script ($1), in paper
lora_alpha = 64 # script ($2), in paper
lora_dropout = 0.05 # default, in paper
lora_target_modules = None # default

# bottleneck adapter hyperparams
bottleneck_size = 256 # default
non_linearity = "tanh" # default
adapter_dropout = 0.0 # default
use_parallel_adapter = False # default
use_adapterp = False # default
target_modules = ["q_proj", "k_proj", "v_proj", "up_proj", "down_proj"] # script, in paper

# Dora hyperparams
dora_simple = True # default
Wdecompose_target_modules = None # default
scaling = 1.0 # default

# prefix tuning hyperparams
num_virtual_tokens = 30 # default

# llm hyperparams
train_on_inputs = True # default
group_by_length = False # default

# wandb params
wandb_project = "" # default
wandb_run_name = "" # default
wandb_watch = "" # default
wandb_log_model = "" # default

# checkpointing
resume_from_checkpoint = None # default

In [8]:
gradient_accumulation_steps = batch_size // micro_batch_size

In [9]:
device_map = "auto"

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    # load_in_8bit=False, -- this is incompatible with latest transformer libraries, should move to BitsAndBytes later
    torch_dtype=torch.bfloat16, # authors used float16
    device_map={"": int(os.environ.get("LOCAL_RANK") or 0)},
    trust_remote_code=True,
)

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [22]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

NameError: name 'AutoTokenizer' is not defined

In [23]:
tokenizer.pad_token_id = (0)

NameError: name 'tokenizer' is not defined

In [ ]:
tokenizer.padding_side = "left"

In [ ]:
## Straight from original authors repo

def tokenize(prompt, add_eos_token=True):
    # there's probably a way to do this with the tokenizer settings
    # but again, gotta move fast
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=cutoff_len,
        padding=False,
        return_tensors=None,
    )
    if (
            result["input_ids"][-1] != tokenizer.eos_token_id
            and len(result["input_ids"]) < cutoff_len
            and add_eos_token
    ):
        result["input_ids"].append(tokenizer.eos_token_id)
        if "chatglm" not in base_model:
            result["attention_mask"].append(1)

    result["labels"] = result["input_ids"].copy()

    if "chatglm" in base_model:
        return {"input_ids": result["input_ids"], "labels": result["labels"]}
    else:
        return result


In [4]:
def generate_prompt(data_point):
    # sorry about the formatting disaster gotta move fast
    if data_point["input"]:
        return f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. 

                ### Instruction:
                {data_point["instruction"]}
                
                ### Input:
                {data_point["input"]}
                
                ### Response:
                {data_point["output"]}""" # noqa: E501
    else:
        return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.  

                ### Instruction:
                {data_point["instruction"]}
                
                ### Response:
                {data_point["output"]}"""

In [ ]:
full_prompt = generate_prompt(data_point)

In [ ]:
tokenized_full_prompt = tokenize(full_prompt)

In [11]:
if adapter_name == "lora":
    config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )
elif adapter_name == "dora":
    print("DoRA init")
    config = DoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        dora_simple=dora_simple,
        Wdecompose_target_modules=Wdecompose_target_modules
    )
a

DoRA init


NameError: name 'DoraConfig' is not defined

In [12]:
model = get_peft_model(model, config)

NameError: name 'get_peft_model' is not defined

In [13]:
data = load_dataset("json", data_files=data_path)

NameError: name 'load_dataset' is not defined

In [14]:
train_val = data["train"].train_test_split(
    test_size=val_set_size, shuffle=True, seed=42
)

train_data = (
    train_val["train"].shuffle().map(generate_and_tokenize_prompt)
)

val_data = (
    train_val["test"].shuffle().map(generate_and_tokenize_prompt)
)


NameError: name 'data' is not defined

In [15]:
trainer = transformers.Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=micro_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=100,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        bf16=True, ## fp16=True
        logging_steps=10,
        optim="adamw_torch",
        evaluation_strategy="steps" if val_set_size > 0 else "no",
        save_strategy="steps",
        eval_steps=eval_step if val_set_size > 0 else None,
        save_steps=save_step,
        output_dir=output_dir,
        save_total_limit=3,
        load_best_model_at_end=True if val_set_size > 0 else False,
        ddp_find_unused_parameters=False if ddp else None,
        group_by_length=group_by_length,
        report_to="wandb" if use_wandb else None,
        run_name=wandb_run_name if use_wandb else None,
    ),
    data_collator=transformers.DataCollatorForSeq2Seq(
        tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
    ),
)


NameError: name 'transformers' is not defined

In [16]:
model.config.use_cache = False

NameError: name 'model' is not defined

In [ ]:
old_state_dict = model.state_dict

In [ ]:
model.state_dict = (
        lambda self, *_, **__: get_peft_model_state_dict(
            self, old_state_dict()
        )
    ).__get__(model, type(model))

In [ ]:
model = torch.compile(model)

In [ ]:
model.save_pretrained(output_dir)